# RAG (Retrieval-Augmented Generation)

RAG is a technique that enhances LLMs by giving them access to external knowledge at inference time, rather than relying solely on what was baked into their weights during training.

---

## The Core Idea

Instead of asking an LLM a question directly, you first **retrieve** relevant documents from an external knowledge base, then **augment** the prompt with that retrieved content, and finally let the model **generate** an answer grounded in that context.

---

## How It Works

### 1.  Ingestion  
Your documents (PDFs, web pages, notes, etc.) are chunked and converted into (vector embeddings, stored in a vector database(Index)) (e.g. Pinecone, FAISS, Chroma).

### 2.  Retrieve
When a user asks a question, the query is also embedded and used to search for the most semantically similar chunks.(The chunkized method is very important; different data format has different methods to chunkize)

### 3.  Augment
The retrieved chunks are injected into the LLM's prompt as context.(if the retrieved chunk is not related to the user's question enough we need to add augment)

### 4.  Generate
The LLM answers the question using both its training knowledge and the retrieved context.

---

## Problems RAG Solves

### 1. Knowledge Cutoff
LLMs are trained on data up to a certain date and can't answer questions about recent events. RAG lets you feed in up-to-date documents at query time.

### 2. Hallucination
LLMs confidently make up facts when they don't know something. By grounding the answer in retrieved documents, RAG gives the model real content to work from instead of guessing.

### 3. No Access to Private / Domain-Specific Data
A general LLM knows nothing about your company's internal docs, your codebase, or your research data. RAG lets you plug in any custom knowledge base without retraining.

### 4. Expensive Fine-Tuning
Teaching an LLM new knowledge by fine-tuning is slow and costly. RAG is a cheaper, faster alternative — just update your document store instead.

### 5. No Source Attribution
Plain LLM outputs give you no way to verify where the answer came from. RAG pipelines can return the source chunks alongside the answer, making responses auditable.

### 6. Context Window Limitations
You can't stuff an entire knowledge base into a prompt. RAG solves this by selectively retrieving only the most relevant chunks, keeping the prompt lean and focused.

# Embeddings & Vector Databases in RAG

---

## 1. Ingestion Pipeline

The **Ingestion Pipeline** is the offline process of preparing your knowledge base before any user query is made.

Source documents are split(text_split)into smaller **chunks**, then each chunk is passed through an **Encoder / Embedding Model**, which converts it into a numerical vector and places it in a **vector space**.

![Embedding](Embedding.jpeg)

The key insight is that **meaning is preserved by proximity** — semantically similar sentences end up close together in that space, regardless of language.

**Example:**
| Sentence | Language |
|---|---|
| "I want to order an extra large coffee" | English |
| "I'll have a tall coffee" | English |
| "Quiero pedir café extra grande" | Spanish |

All three express the same intent, so the embedding model places them **close together** in vector space — even though the third is in a different language entirely.

---

## 2. Indexing

Once all chunks are embedded, their vectors are stored in a **vector database** (e.g. FAISS, Chroma, Pinecone).

The vector database holds the entire knowledge base as a **cloud of points** in high-dimensional space — each point representing one chunk of text, ready to be searched at query time.

---

## 3. Retrieval & Augmentation Pipeline

The **Retrieval Pipeline** is the online process that runs at query time.

![Vector_Database](Vector_Database.jpeg)

### 3.1 Query Encoding
The user's query (e.g. *"What did John do to Alice?"*) is passed through the **same embedding model** used during ingestion, converting it into a vector in the same vector space.

### 3.2 Similarity Search
The query vector is compared against all stored vectors in the database. The **nearest neighbors** — the most semantically relevant chunks — are retrieved.

### 3.3 Augmentation
The retrieved chunks are injected as **context** into the prompt alongside the original query:

```
Prompt:
  Query   = "What did John do to Alice?"
  Context = [chunk 1] [chunk 2] [chunk 3] [chunk 4]
```

---

## 4. Generation

The full augmented prompt is sent to the **LLM**, which generates a grounded and accurate answer based on the retrieved context.

---

## End-to-End RAG Flow

```
[Documents]
     ↓
  Chunking
     ↓
  Embedding Model          ← Ingestion Pipeline (offline)
     ↓
  Vector Database (Index)
     ↑
  Query Embedding          ← Retrieval Pipeline (online)
     ↓
  Similarity Search
     ↓
  Augmented Prompt (Query + Context)
     ↓
   LLM
     ↓
  Answer
```

---

> The magic of embeddings is that **semantic similarity = spatial proximity** in vector space — making it possible to retrieve the right information even across languages or different phrasings.

![RAG_Pipeline](RAG_Pipeline.jpeg)